# Car Price Prediction

**Task:** Predict the price of a car listing based on its characteristics.

**Type:** Regression (supervised learning).

**y:** `price` — the asking price in the listing (not the final transaction price).

**X:** Vehicle characteristics — `brand`, `model`, `year`, `mileage`, `engine`, `fuel`, `gearbox`, `horsepower`, etc.

**Observation:** One car sale listing.

**Metrics:** MAE (primary), RMSE, R². The use of MAPE and/or a log-transformed target will be decided after EDA.

**Success criterion:** Significant improvement over the baseline (median prediction) + an understanding of where and why the model makes errors.

**Known limitations:** The asking price is not the same as the final transaction price. Part of the price variation is driven by seller behavior and is therefore irreducible from the available vehicle characteristics.


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('data/used_cars.csv')

print('DF shape(rows, columns) ', df.shape)
df.head()

DF shape(rows, columns)  (4009, 12)


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [3]:
df.sample(10, random_state=42)

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
2580,Lexus,IS 300 Base,2018,"50,992 mi.",Gasoline,260.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,A/T,White,Gray,At least 1 accident or damage reported,Yes,"$28,000"
3660,Chevrolet,Impala Base,2004,"64,500 mi.",Gasoline,180.0HP 3.4L V6 Cylinder Engine Gasoline Fuel,A/T,Beige,Beige,None reported,Yes,"$5,900"
897,RAM,2500 SLT,2017,"86,000 mi.",Diesel,350.0HP 6.7L Straight 6 Cylinder Engine Diesel...,6-Speed A/T,Gray,Gray,At least 1 accident or damage reported,Yes,"$41,000"
2091,Mercedes-Benz,SL-Class SL 550,2013,"24,933 mi.",Gasoline,429.0HP 4.6L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Silver,Red,At least 1 accident or damage reported,Yes,"$40,250"
1044,Ford,Shelby GT350R Base,2018,"18,500 mi.",Gasoline,526.0HP 5.2L 8 Cylinder Engine Gasoline Fuel,M/T,Blue,Black,At least 1 accident or damage reported,Yes,"$77,999"
2320,GMC,Yukon SLT,2018,"85,500 mi.",Gasoline,355.0HP 5.3L 8 Cylinder Engine Gasoline Fuel,A/T,Black,Black,At least 1 accident or damage reported,Yes,"$35,899"
465,Volvo,V60 Cross Country T5,2020,"10,500 mi.",Gasoline,250.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$36,000"
196,Nissan,Titan XD SV,2020,"10,001 mi.",Gasoline,5.6L V8 32V GDI DOHC,9-Speed Automatic,Black,Black,None reported,Yes,"$47,214"
3113,BMW,330 i,2005,"59,300 mi.",Gasoline,225.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,M/T,Red,Black,None reported,Yes,"$30,900"
3553,Lexus,RX 330 Base,2006,"110,250 mi.",Gasoline,230.0HP 3.3L V6 Cylinder Engine Gasoline Fuel,A/T,White,Beige,None reported,Yes,"$11,000"


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         4009 non-null   str  
 1   model         4009 non-null   str  
 2   model_year    4009 non-null   int64
 3   milage        4009 non-null   str  
 4   fuel_type     3839 non-null   str  
 5   engine        4009 non-null   str  
 6   transmission  4009 non-null   str  
 7   ext_col       4009 non-null   str  
 8   int_col       4009 non-null   str  
 9   accident      3896 non-null   str  
 10  clean_title   3413 non-null   str  
 11  price         4009 non-null   str  
dtypes: int64(1), str(11)
memory usage: 892.6 KB


In [5]:
missing = pd.DataFrame({
    'isna' : df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
})
missing.sort_values('isna',ascending=False)

,isna,pct_missing
clean_title,596,14.87
fuel_type,170,4.24
accident,113,2.82
brand,0,0.00
milage,0,0.00
model_year,0,0.00
model,0,0.00
engine,0,0.00
ext_col,0,0.00
transmission,0,0.00


In [6]:
unique_values = pd.DataFrame(
    {
        'n_unique': df.nunique(),
        'dtypes' : df.dtypes,
    }
).sort_values('n_unique',ascending=False)
unique_values

,n_unique,dtypes
milage,2818,str
model,1898,str
price,1569,str
engine,1146,str
ext_col,319,str
int_col,156,str
transmission,62,str
brand,57,str
model_year,34,int64
fuel_type,7,str


In [7]:
df.describe()

,model_year
count,4009.000000
mean,2015.515590
std,6.104816
min,1974.000000
25%,2012.000000
50%,2017.000000
75%,2020.000000
max,2024.000000


In [8]:
print('Duplicate: ',df.duplicated().sum())
print('Duplicate without price: ',df.drop(columns=['price']).duplicated().sum())

Duplicate:  0
Duplicate without price:  0


In [9]:
SUSPECTS = {'-', '–','—','NA','n/a','N/A','na','unk','unknown','Unknown','None','none','',' ','?',',', 'null','NULL'}
for col in df.columns:
    if df[col].dtype == 'int64':
        continue
    hits = df[col][df[col].isin(SUSPECTS)]
    if len(hits):
        print(f"{col:14s} -> {hits.value_counts().to_dict()}")

fuel_type      -> {'–': 45}
engine         -> {'–': 45}
transmission   -> {'–': 4}
ext_col        -> {'–': 15}
int_col        -> {'–': 133}


In [10]:
for col in ["fuel_type", "accident", "clean_title", "brand"]:
    print(f"\n===== {col} (n_unique={df[col].nunique()}) =====")
    print(df[col].value_counts(dropna=False))


===== fuel_type (n_unique=7) =====
fuel_type
Gasoline          3309
Hybrid             194
NaN                170
E85 Flex Fuel      139
Diesel             116
–                   45
Plug-In Hybrid      34
not supported        2
Name: count, dtype: int64

===== accident (n_unique=2) =====
accident
None reported                             2910
At least 1 accident or damage reported     986
NaN                                        113
Name: count, dtype: int64

===== clean_title (n_unique=1) =====
clean_title
Yes    3413
NaN     596
Name: count, dtype: int64

===== brand (n_unique=57) =====
brand
Ford             386
BMW              375
Mercedes-Benz    315
Chevrolet        292
Porsche          201
Audi             200
Toyota           199
Lexus            163
Jeep             143
Land             130
Nissan           116
Cadillac         107
GMC               91
RAM               91
Dodge             90
Tesla             87
Kia               76
Hyundai           72
Acura           

In [11]:
for col in ["model", "transmission", "ext_col", "engine"]:
    vc = df[col].value_counts()
    print(f"{col:14s} n_unique={len(vc):5d} | "
          f"appear once={(vc == 1).sum():5d} | "
          f"rows covered by top-20={vc.head(20).sum() / len(df):.1%}")

model          n_unique= 1898 | appear once= 1082 | rows covered by top-20=8.0%
transmission   n_unique=   62 | appear once=   22 | rows covered by top-20=96.3%
ext_col        n_unique=  319 | appear once=  214 | rows covered by top-20=88.7%
engine         n_unique= 1146 | appear once=  492 | rows covered by top-20=15.6%


In [12]:
price_num  = pd.to_numeric(df["price"].str.replace(r"[\$,]", "", regex=True),errors="coerce")
milage_num = pd.to_numeric(df["milage"].str.replace(r"[,]|\smi\.", "", regex=True),errors="coerce")

print("price  — failed to parse:", price_num.isna().sum())
print("milage — failed to parse:", milage_num.isna().sum())

pd.DataFrame({"price": price_num, "milage": milage_num}) \
  .describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(0)

price  — failed to parse: 0
milage — failed to parse: 0


,price,milage
count,4009.0,4009.0
mean,44553.0,64718.0
std,78711.0,52297.0
min,2000.0,100.0
1%,4000.0,635.0
25%,17200.0,23044.0
50%,31000.0,52775.0
75%,49990.0,94100.0
95%,111600.0,165000.0
99%,272713.0,222428.0


In [13]:
na_flags = df[["fuel_type", "accident", "clean_title"]].isna()
print("Rows with at least one NaN:", na_flags.any(axis=1).sum())
print("Rows with all three NaN:  ", na_flags.all(axis=1).sum())
na_flags.corr().round(2)

Rows with at least one NaN: 740
Rows with all three NaN:   4


,fuel_type,accident,clean_title
fuel_type,1.00,-0.01,0.00
accident,-0.01,1.00,0.41
clean_title,0.00,0.41,1.00


In [14]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [15]:
data = df.copy()

PLACEHOLDERS = ['–', '—', '-', 'not supported',  'unknown', 'Unknown', 'n/a', 'N/A', '']
obj_cols = data.columns[data.dtypes != "int64"]

na_before = data[obj_cols].isna().sum()
data[obj_cols] = data[obj_cols].replace(PLACEHOLDERS, np.nan)
na_after = data[obj_cols].isna().sum()

pd.DataFrame({
    'na_before': na_before,
    'na_after': na_after,
    'added': na_after - na_before,
}).sort_values("added", ascending=False)

,na_before,na_after,added
int_col,0,133,133
fuel_type,170,217,47
engine,0,45,45
ext_col,0,15,15
transmission,0,4,4
brand,0,0,0
model,0,0,0
milage,0,0,0
accident,113,113,0
clean_title,596,596,0


In [16]:
for col in ["transmission", "ext_col", "int_col", "fuel_type"]:
    short = data[col].dropna()
    short = short[short.str.len() <= 3]
    print(f"\n===== {col} =====")
    print(short.value_counts())


===== transmission =====
transmission
A/T    1037
M/T      40
2         3
F         2
Name: count, dtype: int64

===== ext_col =====
ext_col
Red    261
Blu      3
Tan      2
Ice      1
Name: count, dtype: int64

===== int_col =====
int_col
Red    126
Tan      4
Blk      3
Ice      2
Ash      1
Name: count, dtype: int64

===== fuel_type =====
Series([], Name: count, dtype: int64)


In [17]:
data.loc[data['transmission'].isin(['2','F']), 'transmission'] = np.nan

COLOR_FIX = {'Blu' : 'Blue', 'Blk' : 'Black'}
data['ext_col'] = data['ext_col'].replace(COLOR_FIX)
data['int_col'] = data['int_col'].replace(COLOR_FIX)

print("transmission NaN:", data["transmission"].isna().sum())

transmission NaN: 9


In [18]:
data['price'] = pd.to_numeric(data["price"].str.replace(r"[\$,]", "", regex=True), errors="coerce")
data['milage'] = pd.to_numeric(data["milage"].str.replace(r"[,]|\smi\.", "", regex=True), errors="coerce")

print('Price NAN: ', data["price"].isna().sum())
print('Milage NAN: ', data["milage"].isna().sum())

print('-----')
print(data['price'].head(5))
print(data['milage'].head(5))

Price NAN:  0
Milage NAN:  0
-----
0    10300
1    38005
2    54598
3    15500
4    34999
Name: price, dtype: int64
0    51000
1    34742
2    22372
3    88900
4     9835
Name: milage, dtype: int64


In [19]:
eng = data["engine"]

data["horsepower"]    = eng.str.extract(r"(\d+\.?\d*)\s*HP").astype(float)
data["engine_volume"] = eng.str.extract(r"(\d\.\d)\s*(?:L\b|Liter)").astype(float)
data["cylinders"]     = (eng.str.extract(r"(?:V|I|Straight\s)(\d+)|(\d+)\s*Cylinder").bfill(axis=1).iloc[:, 0].astype(float))

data["is_hybrid"]   = eng.str.contains(r"Hybrid|Gas/Electric", case=False, na=False)
data["is_electric"] = eng.str.contains("Electric", case=False, na=False) & ~data["is_hybrid"]

print("hybrid:", data["is_hybrid"].sum(), "| electric:", data["is_electric"].sum())
data[["engine", "horsepower", "engine_volume", "cylinders"]].head(8)

hybrid: 162 | electric: 202


,engine,horsepower,engine_volume,cylinders
0,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,300.0,3.7,6.0
1,3.8L V6 24V GDI DOHC,NaN,3.8,6.0
2,3.5 Liter DOHC,NaN,3.5,NaN
3,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,354.0,3.5,6.0
4,2.0L I4 16V GDI DOHC Turbo,NaN,2.0,4.0
5,2.4 Liter,NaN,2.4,NaN
6,292.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,292.0,2.0,4.0
7,282.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,282.0,4.4,8.0


In [20]:
tr = data["transmission"]
data["n_speeds"] = tr.str.extract(r"(\d+)[-\s]?Speed").astype(float)

def gearbox_type(s):
    if pd.isna(s):
        return np.nan
    s = s.lower()
    if "cvt" in s or "variable" in s:    return "CVT"
    if "dual shift" in s or "dct" in s:  return "DCT"
    if "m/t" in s or "manual" in s:      return "Manual"
    if "a/t" in s or "automatic" in s:   return "Automatic"
    return "Other"

data["gearbox"] = tr.apply(gearbox_type)
print(data["gearbox"].value_counts(dropna=False))

gearbox
Automatic    3108
DCT           399
Manual        374
CVT           104
Other          15
NaN             9
Name: count, dtype: int64


In [21]:
new_cols = ["horsepower", "engine_volume", "cylinders", "n_speeds"]
pd.DataFrame({
    "n_missing": data[new_cols].isna().sum(),
    "pct_missing": (data[new_cols].isna().mean() * 100).round(1),
    "min": data[new_cols].min(),
    "median": data[new_cols].median(),
    "max": data[new_cols].max(),
})

,n_missing,pct_missing,min,median,max
horsepower,808,20.2,70.0,310.0,1020.0
engine_volume,243,6.1,1.0,3.5,8.4
cylinders,440,11.0,3.0,6.0,12.0
n_speeds,1853,46.2,1.0,7.0,10.0


In [22]:
data.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price,horsepower,engine_volume,cylinders,is_hybrid,is_electric,n_speeds,gearbox
0,Ford,Utility Police Interceptor Base,2013,51000,E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,10300,300.0,3.7,6.0,False,False,6.0,Automatic
1,Hyundai,Palisade SEL,2021,34742,Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,38005,NaN,3.8,6.0,False,False,8.0,Automatic
2,Lexus,RX 350 RX 350,2022,22372,Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,54598,NaN,3.5,NaN,False,False,NaN,Automatic
3,INFINITI,Q50 Hybrid Sport,2015,88900,Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,15500,354.0,3.5,6.0,True,False,7.0,Automatic
4,Audi,Q3 45 S line Premium Plus,2021,9835,Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,34999,NaN,2.0,4.0,False,False,8.0,Automatic


In [23]:
data.groupby("gearbox", dropna=False)["n_speeds"].agg(
    n_rows="size",
    n_missing=lambda s: s.isna().sum(),
    pct_missing=lambda s: round(s.isna().mean() * 100, 1),
)

,n_rows,n_missing,pct_missing
gearbox,,,
Automatic,3108,1285,41.3
CVT,104,104,100.0
DCT,399,398,99.7
Manual,374,46,12.3
Other,15,11,73.3
NaN,9,9,100.0


In [24]:
nan_fuel = data[data["fuel_type"].isna()]
print("rows:", len(nan_fuel))
print(nan_fuel["brand"].value_counts().head(8))
print()
print(nan_fuel["engine"].value_counts().head(5))

rows: 217
brand
Tesla        87
Rivian       17
Ford         17
Porsche      11
Chevrolet     8
Nissan        8
Dodge         8
Toyota        6
Name: count, dtype: int64

engine
Electric                                       17
835.0HP Electric Motor Electric Fuel System    16
425.0HP Electric Motor Electric Fuel System    16
455.0HP Electric Motor Electric Fuel System    12
518.0HP Electric Motor Electric Fuel System    11
Name: count, dtype: int64


In [25]:
data.sample(10)

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price,horsepower,engine_volume,cylinders,is_hybrid,is_electric,n_speeds,gearbox
2653,Land,Rover Range Rover HSE SWB,2020,57785,Hybrid,395.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,A/T,Black,Black,None reported,Yes,63995,395.0,3.0,6.0,True,False,NaN,Automatic
3173,Subaru,BRZ Limited,2013,46452,Gasoline,200.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,6-Speed M/T,Gray,Black,None reported,Yes,22888,200.0,2.0,4.0,False,False,6.0,Manual
1634,Cadillac,Escalade Platinum,2019,92000,Gasoline,420.0HP 6.2L 8 Cylinder Engine Gasoline Fuel,A/T,Black,Black,None reported,Yes,51900,420.0,6.2,8.0,False,False,NaN,Automatic
2511,Porsche,718 Boxster Base,2017,26000,Gasoline,300.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,A/T,Gray,Black,None reported,Yes,45500,300.0,2.0,4.0,False,False,NaN,Automatic
3834,Ford,F-150 Lightning LARIAT,2022,470,NaN,563.0HP Electric Motor Electric Fuel System,1-Speed A/T,White,Gray,None reported,Yes,71900,563.0,NaN,NaN,False,True,1.0,Automatic
513,BMW,Z4 2.5i Roadster,2005,138000,Gasoline,184.0HP 2.5L Straight 6 Cylinder Engine Gasoli...,5-Speed A/T,White,Beige,None reported,Yes,9950,184.0,2.5,6.0,False,False,5.0,Automatic
1391,Mercedes-Benz,Metris Base,2019,10500,Gasoline,208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,A/T,Black,Brown,None reported,Yes,79500,208.0,2.0,4.0,False,False,NaN,Automatic
103,BMW,i8 Base,2019,41500,Plug-In Hybrid,369.0HP 1.5L 3 Cylinder Engine Plug-In Electri...,Transmission w/Dual Shift Mode,White,Black,None reported,Yes,86000,369.0,1.5,3.0,False,True,NaN,DCT
2520,Genesis,G70 3.3T Advanced,2019,48000,Gasoline,365.0HP 3.3L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,None reported,Yes,29999,365.0,3.3,6.0,False,False,8.0,Automatic
1751,Land,Rover Range Rover HSE,2012,102800,Gasoline,385.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,Gold,Brown,None reported,Yes,16000,385.0,5.0,8.0,False,False,NaN,Automatic


In [26]:
electric_mask = data['engine'].str.contains(r'Electric|Dual Motor|Battery|kW', case=False, na=False)
data.loc[data['fuel_type'].isna() & electric_mask, "fuel_type"] = 'Electric'
print(data['fuel_type'].value_counts(dropna=False))

fuel_type
Gasoline          3309
Hybrid             194
Electric           172
E85 Flex Fuel      139
Diesel             116
NaN                 45
Plug-In Hybrid      34
Name: count, dtype: int64


In [27]:
data["has_fixed_gears"] = ~data["gearbox"].isin(["CVT"])
data["has_combustion"]  = ~data["fuel_type"].eq("Electric")

n_speeds_na = data["n_speeds"].isna()
print("not applicable (CVT):", (n_speeds_na & ~data["has_fixed_gears"]).sum())
print("unknown (A/T, DCT):  ", (n_speeds_na &  data["has_fixed_gears"]).sum())

not applicable (CVT): 104
unknown (A/T, DCT):   1749
